In [12]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub
from pathlib import Path



In [13]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [14]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/tmp/ipykernel_1052/93639238.py:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  text = re.sub('\s+', ' ', text).strip()


In [15]:
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [16]:
df['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [17]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [18]:
df["sentiment"] = (
    df["sentiment"]
    .replace({"sadness": 0, "happiness": 1})
    .astype("int64")
)
df.head()

,sentiment,content
1,0,layin n bed headache ughhhh waitin call
2,0,funeral ceremony gloomy friday
6,0,sleep im not thinking old friend want married ...
8,0,charviray charlene love miss
9,0,kelcouch sorry least friday


In [19]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
import dagshub

dagshub.init(repo_owner='VrajPatel105', repo_name='mlflow-mini-project', mlflow=True)

mlflow.set_tracking_uri("https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow")

Initialized MLflow to track repo "VrajPatel105/mlflow-mini-project"

Repository VrajPatel105/mlflow-mini-project initialized!

In [22]:
# Set the experiment name
mlflow.set_experiment("LR Hyperparameter Tuning")


# Define hyperparameter grid for Logistic Regression
param_grid = {
    "C": [0.1, 1, 10],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear"],
}


# Parent run: contains the overall tuning summary and best model
with mlflow.start_run(run_name="Logistic Regression Grid Search"):

    # Perform grid search using training data only
    grid_search = GridSearchCV(
        estimator=LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),
        param_grid=param_grid,
        cv=5,
        scoring="f1",
        n_jobs=-1,
        refit=True,
    )

    grid_search.fit(X_train, y_train)

    # Log each parameter combination as a child run
    for params, mean_score, std_score in zip(
        grid_search.cv_results_["params"],
        grid_search.cv_results_["mean_test_score"],
        grid_search.cv_results_["std_test_score"],
    ):
        with mlflow.start_run(
            run_name=f"LR | C={params['C']} | penalty={params['penalty']}",
            nested=True,
        ):
            # Train one model with this exact parameter combination
            model = LogisticRegression(
                **params,
                max_iter=1000,
                random_state=42,
            )

            model.fit(X_train, y_train)

            # Evaluate on the untouched test set
            y_pred = model.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred, zero_division=0)
            recall = recall_score(y_test, y_pred, zero_division=0)
            f1 = f1_score(y_test, y_pred, zero_division=0)

            # Log the parameter combination and its CV/test results
            mlflow.log_params(params)

            mlflow.log_metrics({
                "mean_cv_f1": mean_score,
                "std_cv_f1": std_score,
                "test_accuracy": accuracy,
                "test_precision": precision,
                "test_recall": recall,
                "test_f1": f1,
            })

            print(f"\nParameters: {params}")
            print(f"Mean CV F1: {mean_score:.4f}")
            print(f"Std CV F1: {std_score:.4f}")
            print(f"Test Accuracy: {accuracy:.4f}")
            print(f"Test Precision: {precision:.4f}")
            print(f"Test Recall: {recall:.4f}")
            print(f"Test F1: {f1:.4f}")

    # Best result selected from cross-validation only
    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    best_cv_f1 = grid_search.best_score_

    # Evaluate the chosen model once on test data
    best_y_pred = best_model.predict(X_test)

    best_test_accuracy = accuracy_score(y_test, best_y_pred)
    best_test_precision = precision_score(
        y_test,
        best_y_pred,
        zero_division=0,
    )
    best_test_recall = recall_score(
        y_test,
        best_y_pred,
        zero_division=0,
    )
    best_test_f1 = f1_score(
        y_test,
        best_y_pred,
        zero_division=0,
    )

    # Log the winning configuration in the parent run
    mlflow.log_params({
        f"best_{key}": value
        for key, value in best_params.items()
    })

    mlflow.log_metrics({
        "best_cv_f1": best_cv_f1,
        "best_test_accuracy": best_test_accuracy,
        "best_test_precision": best_test_precision,
        "best_test_recall": best_test_recall,
        "best_test_f1": best_test_f1,
    })

    # Log the best trained Logistic Regression model
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="best_model",
    )

    # Log the notebook only if the file exists in the current directory
    notebook_path = Path.cwd() / "experiment3_lr_hyperparameter_tuning.ipynb"

    print(f"\nNotebook path: {notebook_path}")
    print(f"Notebook exists: {notebook_path.exists()}")

    if notebook_path.exists():
        mlflow.log_artifact(str(notebook_path))
    else:
        print(
            "Notebook artifact not logged: "
            f"file not found at {notebook_path}"
        )

    print("\nBest Parameters:", best_params)
    print(f"Best Cross-Validation F1: {best_cv_f1:.4f}")
    print(f"Best Test Accuracy: {best_test_accuracy:.4f}")
    print(f"Best Test Precision: {best_test_precision:.4f}")
    print(f"Best Test Recall: {best_test_recall:.4f}")
    print(f"Best Test F1: {best_test_f1:.4f}")

2026/09/10 00:02:29 INFO mlflow.tracking.fluent: Experiment with name 'LR Hyperparameter Tuning' does not exist. Creating a new experiment.
/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead


Parameters: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
Mean CV F1: 0.7052
Std CV F1: 0.0142
Test Accuracy: 0.7398
Test Precision: 0.7752
Test Recall: 0.6591
Test F1: 0.7125
🏃 View run LR | C=0.1 | penalty=l1 at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5/runs/a26ea1b1bcb648cb980257cc06ca7f05
🧪 View experiment at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5


/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Mean CV F1: 0.7750
Std CV F1: 0.0113
Test Accuracy: 0.7807
Test Precision: 0.7740
Test Recall: 0.7793
Test F1: 0.7766
🏃 View run LR | C=0.1 | penalty=l2 at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5/runs/e8f7dfee5d2c43e391b829b3716755d9
🧪 View experiment at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5


/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


...


Parameters: {'C': 1, 'penalty': 'l2', 'solver': 'liblinear'}
Mean CV F1: 0.7768
Std CV F1: 0.0090
Test Accuracy: 0.7740
Test Precision: 0.7650
Test Recall: 0.7764
Test F1: 0.7707
🏃 View run LR | C=1 | penalty=l2 at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5/runs/aa6642f5918d40ae9b668e4ab918ce58
🧪 View experiment at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5


/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



Parameters: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
Mean CV F1: 0.7683
Std CV F1: 0.0142
Test Accuracy: 0.7643
Test Precision: 0.7534
Test Recall: 0.7704
Test F1: 0.7618
🏃 View run LR | C=10 | penalty=l1 at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5/runs/7afb5a95cfbe4d1f9c46dfcc7de7314c
🧪 View experiment at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5


/mnt/c/dev/projects/mlops-mini-project/tempenv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(



Parameters: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
Mean CV F1: 0.7708
Std CV F1: 0.0129
Test Accuracy: 0.7634
Test Precision: 0.7534
Test Recall: 0.7675
Test F1: 0.7604
🏃 View run LR | C=10 | penalty=l2 at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5/runs/8d80036ea6e84a129aa98d12f4421a64
🧪 View experiment at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/experiments/5

Notebook path: /mnt/c/dev/projects/mlops-mini-project/notebooks/experiment3_lr_hyperparameter_tuning.ipynb
Notebook exists: False
Notebook artifact not logged: file not found at /mnt/c/dev/projects/mlops-mini-project/notebooks/experiment3_lr_hyperparameter_tuning.ipynb

Best Parameters: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}
Best Cross-Validation F1: 0.7812
Best Test Accuracy: 0.7836
Best Test Precision: 0.7785
Best Test Recall: 0.7793
Best Test F1: 0.7789
🏃 View run Logistic Regression Grid Search at: https://dagshub.com/VrajPatel105/mlflow-dagshub.mlflow/#/exp